# Skill extraction rules (spaCy EntityRuler)


TIER 1 - PHRASES (any case)  
        Long / distinctive names. "projektledning", "Projektledning" and "PROJEKTLEDNING" all match. This is the default tier.  
        Example: "Kravhantering", "Docker".
 
TIER 2 - EXACT CASE (short or ambiguous words)
    Short acronyms and words that are also ordinary words.
    Only the exact spelling matches. "SAP" matches, "sap" does not.  
    Example: "R", "SAP", "REST".
 
TIER 3 - CUSTOM ALIASES (synonyms -> one canonical skill)
    Different spellings / languages / abbreviations that should all map to ONE canonical name. Each alias is declared as any-case or exact-case, because "Machine learning" is safe in any case but "ML" and "AI" are not.


In [1]:
import spacy
import json
import pandas as pd

## Specific for me
**Allmänna datakunskaper**:  
    Systemanalys, Systemarkitektur, Verksamhetsutveckling, Systemintegration, Cybersäkerhet   

**Applikationsplattformar**:  
    Kubernetes, Red Hat OpenShift, Oracle Cloud Infrastructure (OCI), Oracle Service Bus, Oracle SOA Suite, MuleSoft Anypoint Platform 

**Certifikat/licenser**:    
    Certified Business Analysis Professional (CBAP), Entry Certificate in Business Analysis (ECBA), TOGAF-certifiering, AWS Certified Solutions Architect, Microsoft Certified: Azure Solutions Architect Expert

In [2]:
# TIER 1 - PHRASES, matched in ANY case
# Taxonomy labels, copied as-is.

TIER1_PHRASES = [
"Kravhantering", 
"Systemanalys", # New
"Systemarkitektur", # New
"Systemintegration", # New
"Verksamhetsutveckling", # New
"Cybersäkerhet", # New
"Oracle SOA Suite", # New 
"Docker",
"IBM Cloud", 
"Sun Java Enterprise System",
"Windows Server", 
"Associate Android Developer",
"Oracle Certified Associate",
"PHP Developer Certification",
"AWS Certified Solutions Architect" # New
]


# TIER 2 - EXACT CASE, short or ambiguous words
# {canonical skill: [exact spellings]}

TIER2_EXACT_CASE = {
    "A+": ["A+"],
    "Odoo": ["Odoo"],
    "SAP": ["SAP"],
}


# TIER 3 - CUSTOM ALIASES
# {canonical skill: {"any_case": [...], "exact_case": [...]}}

TIER3_CUSTOM = {
    "Internet Information Services": {
        "any_case": ["Internet Information Services", "Internet Information Services (IIS)"],
        "exact_case": ["IIS"],
    },
    "Oracle Cloud Infrastructure": {
        "any_case": ["Oracle Cloud Infrastructure", "Oracle Cloud Infrastructure (OCI)", "Oracle Cloud"],
        "exact_case": ["OCI"],
    }, # New
    "Amazon Web Services": {
        "any_case": ["Amazon Web Services", "Amazon Web Services (AWS)"],
        "exact_case": ["AWS"],
    },
    "Infor": {
        "any_case": ["Infor OS", "Infor CloudSuite"],
        "exact_case": ["Infor"],
    },
    "Microsoft Dynamics 365": {
      "any_case": ["Microsoft Dynamics 365", "Dynamics 365"],
      "exact_case": ["D365"],
  },
    "Office SharePoint Server": {
        "any_case": ["Office SharePoint Server", "Office SharePoint Server (MOSS)", "SharePoint Server", "SharePoint", "SharePoint Online"],
        "exact_case": ["MOSS"],
    },
    "Oracle JCAPS": {
        "any_case": ["Oracle JCAPS", "Oracle Java CAPS", "Java Composite Application Platform Suite"],
        "exact_case": ["Java CAPS"],
    }, # this was discontinued in 2016
    "Oracle Service Bus": {
        "any_case": ["Oracle Service Bus", "Oracle Service Bus (OSB)"],
        "exact_case": ["OSB"],
    }, # New
    "Oracle NetSuite": {
        "any_case": ["Oracle NetSuite", "NetSuite"],
    },
    "Salesforce Cloud": {
        "any_case": ["Salesforce Cloud", "Service Cloud", "Sales Cloud", "Salesforce"],
    },
    "MuleSoft Anypoint Platform": {
        "any_case": ["MuleSoft Anypoint Platform", "Mulesoft-plattformen", "MuleSoft"],
    }, # New
    "Red Hat OpenShift": {
        "any_case": ["Red Hat OpenShift", "OpenShift"],
    }, # New
    "AZ-204: Developing Solutions for Microsoft Azure": {
        "any_case": ["AZ-204: Developing Solutions for Microsoft Azure", "Developing Solutions for Microsoft Azure"],
        "exact_case": ["AZ-204"],
    },
    "AZ-900: Microsoft Azure Fundamentals": {
        "any_case": ["AZ-900: Microsoft Azure Fundamentals", "Microsoft Azure Fundamentals"],
        "exact_case": ["AZ-900"],
    },
    "MB-600: Microsoft Dynamics 365 Power Platform Solution Architect": {
        "any_case": ["MB-600: Microsoft Dynamics 365 Power Platform Solution Architect", "Dynamics 365 + Power Platform Solution Architect Expert"],
        "exact_case": ["MB-600"],
    },
    "Microsoft 365 Certified: Fundamentals": {
        "any_case": ["Microsoft 365 Certified: Fundamentals", "Microsoft Certified: Fundamentals"],
        "exact_case": ["MS-900"],
    },
    "Microsoft Certified: Application Developer": {
        "any_case": ["Microsoft Certified: Application Developer", "Microsoft Certified: Application Developer (MCAD)"],
        "exact_case": ["MCAD"],
    },
    "Microsoft Certified: Database Administrator": {
        "any_case": ["Microsoft Certified: Database Administrator", "Microsoft Certified: Database Administrator (MCDBA)"],
        "exact_case": ["MCDBA"],
    },
    "Microsoft Certified: Professional": {
        "any_case": ["Microsoft Certified: Professional", "Microsoft Certified: Professional (MCP)"],
    },
    "Microsoft Certified: Solution Developer": {
        "any_case": ["Microsoft Certified: Solution Developer", "Microsoft Certified: Solution Developer (MCSD)"],
        "exact_case": ["MCSD"],
    },
    "Microsoft Certified: System Engineer": {
        "any_case": ["Microsoft Certified: System Engineer", "Microsoft Certified: System Engineer (MCSE)"],
        "exact_case": ["MCSE"],
    },
    "Microsoft Certified: Azure Administrator Associate": {
        "any_case": ["Microsoft Certified: Azure Administrator Associate"],
    },
    "Microsoft Certified: Azure Solutions Architect Expert": {
        "any_case": ["Microsoft Certified: Azure Solutions Architect Expert", "Azure Solutions Architect Expert"],
    }, # New
    "Oracle Certified Database Administrator": {
        "any_case": ["Oracle Certified Database Administrator", "Oracle Certified Database Administrator (OCDBA)"],
        "exact_case": ["OCDBA"],
    },
    "Oracle Certified Database Operator": {
        "any_case": ["Oracle Certified Database Operator", "Oracle Certified Database Operator (DBO)"],
        "exact_case": ["DBO"],
    },
    "Oracle Certified Professional": {
        "any_case": ["Oracle Certified Professional", "Oracle Certified Professional/OCP", "Oracle Certified Professional (OCP)"],
    },
    "Certified Business Analysis Professional": {
        "any_case": ["Certified Business Analysis Professional", "Certified Business Analysis Professional (CBAP)"],
        "exact_case": ["CBAP"],
    }, # New
    "Entry Certificate in Business Analysis": {
        "any_case": ["Entry Certificate in Business Analysis", "Entry Certificate in Business Analysis (ECBA)"],
        "exact_case": ["ECBA"],
    }, # New
    "TOGAF-certifiering": {
        "any_case": ["TOGAF-certifiering"],
        "exact_case": ["TOGAF"],
    }, # New
    "Certified JavaScript Developer": {
        "any_case": ["Certified JavaScript Developer", "JavaScript Developer-certifiering"],
    }, # New
    "Windows Azure": {
        "any_case": ["Microsoft Azure", "Windows Azure"],
    },
    "Google Cloud Platform": {
        "any_case": ["Google Cloud", "Google Cloud Platform"],
        "exact_case": ["GCP"]
    },
    "Kubernetes": {
        "any_case": ["Kubernetes"],
        "exact_case": ["K8s"]
    }, # New
    "Apache Kafka": {
        "any_case": ["Apache Kafka", "Kafka"],
    },
    "Apache Tomcat": {
        "any_case": ["Apache Tomcat", "Tomcat"],
    },
    "Weblogic Server": {
        "any_case": ["Weblogic Server", "WebLogic", "Oracle WebLogic"],
    },
    "JBoss Application Server": {
        "any_case": ["JBoss Application Server", "JBoss", "JBoss EAP"],
    },
    "IBM WebSphere": {
        "any_case": ["IBM WebSphere", "WebSphere"],
    },
    "Glassfish Server": {
        "any_case": ["Glassfish Server", "GlassFish"],
    },
    "IBM Domino": {
        "any_case": ["IBM Domino", "HCL Domino", "Lotus Domino"],
    },
 } 

In [3]:
LABEL = "SKILL"

# Blank Swedish pipeline (tokenizer + ruler)

nlp = spacy.blank("sv")
ruler = nlp.add_pipe("entity_ruler")

# 3. Helpers: turn a string into a token pattern
# We tokenize with the SAME tokenizer the pipeline uses,
# so the pattern always lines up with how the text is split.

def any_case_pattern(text, skill):
    tokens = [{"LOWER": t.lower_} for t in nlp.make_doc(text)]
    return {"label": LABEL, "pattern": tokens, "id": skill}

def exact_case_pattern(text, skill):
    tokens = [{"ORTH": t.text} for t in nlp.make_doc(text)]
    return {"label": LABEL, "pattern": tokens, "id": skill}

# 4. Build all patterns from the three tiers

patterns = []

# Tier 1: the phrase is its own canonical name, matched in any case
for phrase in TIER1_PHRASES:
    patterns.append(any_case_pattern(phrase, skill=phrase))

# Tier 2: short/ambiguous words, exact spelling only
for skill, spellings in TIER2_EXACT_CASE.items():
    for spelling in spellings:
        patterns.append(exact_case_pattern(spelling, skill))

# Tier 3: custom aliases, each list has its own matching rule
for skill, aliases in TIER3_CUSTOM.items():
    for text in aliases.get("any_case", []):
        patterns.append(any_case_pattern(text, skill))
    for text in aliases.get("exact_case", []):
        patterns.append(exact_case_pattern(text, skill))

ruler.add_patterns(patterns)
print(f"Added {len(patterns)} patterns")

Added 130 patterns


In [4]:
# 5. Helper to extract skills from a job ad

def extract_skills(text):
    doc = nlp(text)
    # ent.ent_id_ is the canonical skill name, ent.text is what was found
    return sorted({ent.ent_id_ for ent in doc.ents if ent.label_ == LABEL})

# 6. Try it

ad = """
Vi söker en konsult med erfarenhet av kravhantering och systemanalys.
Du kan AWS, docker och Azure. Kunskap om SAP är meriterande.
"""

doc = nlp(ad)
for ent in doc.ents:
    print(f"{ent.text} -> {ent.ent_id_}")

print(extract_skills(ad))

kravhantering -> Kravhantering
systemanalys -> Systemanalys
AWS -> Amazon Web Services
docker -> Docker
SAP -> SAP
['Amazon Web Services', 'Docker', 'Kravhantering', 'SAP', 'Systemanalys']


In [5]:
with open("../../data/train_set.json", "r", encoding="utf-8") as file:
    ads = json.load(file)

df = pd.DataFrame(ads)

In [6]:
def extract_skills_batch(texts):
    results = []
    for doc in nlp.pipe(texts, batch_size=100):
        skills = sorted({ent.ent_id_ for ent in doc.ents if ent.label_ == LABEL})
        results.append(skills)
    return results

df["skills"] = extract_skills_batch(df["description"])

In [7]:
df[["description", "skills"]]

,description,skills
0,Har du erfarenhet av Dynamics 365 och/eller AX...,[Microsoft Dynamics 365]
1,"Omvärlden förändras i snabb takt. Ny teknik, n...",[Verksamhetsutveckling]
2,Är du en analytisk och lösningsorienterad pers...,"[Kubernetes, Red Hat OpenShift]"
3,Vill du arbeta med den senaste tekniken och hj...,[Microsoft Dynamics 365]
4,Lantmäteriet är den myndighet som kartlägger S...,[]
...,...,...
3076,Vi växer och söker en ny medlem till vårt ente...,[]
3077,Åt vår kund söker vi en Kravledare\n\nUppdrags...,[]
3078,Är du konsult / interim och söker uppdrag som ...,[]
3079,Vi söker en IT-arkitekt för kundsräkning som s...,[]


In [8]:
# How many skills were found per ad?
df["n_skills"] = df["skills"].str.len()
print(df["n_skills"].describe())
print()

# Most common skills across all ads
skill_counts = df["skills"].explode().value_counts()
print(skill_counts.head(30))

count    3081.000000
mean        0.654008
std         0.900788
min         0.000000
25%         0.000000
50%         0.000000
75%         1.000000
max         7.000000
Name: n_skills, dtype: float64

skills
Verksamhetsutveckling                                    259
Kravhantering                                            256
Kubernetes                                               162
Microsoft Dynamics 365                                   150
TOGAF-certifiering                                       134
Systemarkitektur                                         126
Amazon Web Services                                      122
Cybersäkerhet                                            100
SAP                                                       95
Docker                                                    91
Systemintegration                                         90
Red Hat OpenShift                                         82
Windows Azure                                             80


In [9]:
skill = "Cybersäkerhet"
skill_ads = df[df["skills"].apply(lambda skills: skill in skills)]

for _, row in skill_ads.head(5).iterrows():
    print("TITLE:", row["title"])
    text = row["description"]

    for ent in nlp(text).ents:
        if ent.ent_id_ == skill:
            start = ent.start_char
            print(ent.text, "->", ent.ent_id_)
            print(text[max(0, start - 50): start + 50])
            print("-" * 80)

TITLE: IT-Arkitekt inom IAM
cybersäkerhet -> Cybersäkerhet
 I rollen kommer fokus vara på identitetssäkerhet/cybersäkerhet, behörighetskontroll och livscykelha
--------------------------------------------------------------------------------
TITLE: Solution Architect
cybersäkerhet -> Cybersäkerhet
i söker en mycket erfaren Solution Architect inom cybersäkerhet till ett tekniskt komplext uppdrag m
--------------------------------------------------------------------------------
TITLE: Systemingenjör Härjeåns
cybersäkerhet -> Cybersäkerhet
en är det meriterande. Du behöver ha intresse för cybersäkerhet och mjukvaruutveckling samt goda kun
--------------------------------------------------------------------------------
TITLE: Senior Solution Architect  sökes till Iver
cybersäkerhet -> Cybersäkerhet
serade IT-tjänster med spetskompetens inom drift, cybersäkerhet, cloud och innovation. Våra tjänster
--------------------------------------------------------------------------------
TITLE: IT-Ark

# Limits
- No context in the matchings
    - Possible solution: writing more custom rules to also check the words around the detected skill
- All based on tokenization, no lemmatization
    